&nbsp;

&nbsp;

# Enhancer-derived lncRNA (up5p)

- enhancer の近傍にある lncRNA を調査する
- [ENCODE-rE2G](https://github.com/EngreitzLab/ENCODE_rE2G) の予測結果に該当する遺伝子を調査する
- これまでの定義では、紐づく fanta.bio の CRE が `ELA` であることを条件としていたが、enhancer から転写されるとされる `ENEMAL` の近傍にある CRE が `PLA` であり、見逃していることに気づいた
- このような背景から、[ENCODE-rE2G](https://github.com/EngreitzLab/ENCODE_rE2G) の予測結果とオーバーラップするデータを、enhancer-derived lncRNA として調査する
- [../05_ENCODE_rE2G/encode_re2g_parse.ipynb](../05_ENCODE_rE2G/encode_re2g_parse.ipynb)で処理した結果と、up5p transcript list を `join`する

In [1]:
import datetime
import os
import pathlib

import polars as pl
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

&nbsp;

&nbsp;

## 1. e2g dataset (Heat stress)

- `TargetGene` の近傍ではない enhancer を標的とする
- `isSelfPromoter` == False でフィルタリングする

In [2]:
e2g_join_filter = pl.read_parquet(
    "../../out/ENCODE_rE2G/parquet_file/encode_re2g_hs_join_v261003.parquet"
).filter(
    pl.col("isSelfPromoter") == False
)

display(e2g_join_filter)

ENCODE-rE2G.Score,re2g_dataset,cre_id,cre_name,cre_chr,fantabio_cre_class,enhancer_cre_overlap_bp,enhancer_cre_overlap_frac_of_fantabio_cre,TargetGene,target_cre_id,target_cre_name,target_fantabio_cre_class,class,isSelfPromoter,distanceToTSS,re2g_enhancer_locus,fantabio_enhancer_cre,re2g_features,target_tss_info,target_cre_info,target_cre_overlap_info
f64,str,str,str,str,str,i64,f64,str,str,str,str,str,bool,f64,struct[3],struct[3],struct[8],struct[5],struct[10],struct[2]
0.342709,"""GSE154744_K562_42C_25min""","""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",353,0.842482,"""AZIN2""","""FCHS_9997""","""cp4@AZIN2""","""ELA""","""genic""",false,15223.0,"{""chr1"",33097150,33097650}","{33097084,33097503,0.614679}","{33082177,""ENSG00000142920"",0.383964,null,0.044833,2,1,220.597881}","{""chr1"",33081927,33082427,0,""+""}","{""chr1"",33081878,33082317,0,""."",33081878,33082179,""255,202,74"",""directionality:1|class:ELA"",1.0}","{390,0.78}"
0.320048,"""GSE154744_K562_42C_60min""","""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",288,0.687351,"""AZIN2""","""FCHS_9997""","""cp4@AZIN2""","""ELA""","""genic""",false,15288.0,"{""chr1"",33097215,33097715}","{33097084,33097503,0.614679}","{33082177,""ENSG00000142920"",0.402521,null,0.044762,4,1,253.0904}","{""chr1"",33081927,33082427,0,""+""}","{""chr1"",33081878,33082317,0,""."",33081878,33082179,""255,202,74"",""directionality:1|class:ELA"",1.0}","{390,0.78}"
0.312414,"""GSE267931_HUVEC_39C_120min""","""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",358,0.854415,"""AZIN2""","""FCHS_9997""","""cp4@AZIN2""","""ELA""","""genic""",false,15218.0,"{""chr1"",33097145,33097645}","{33097084,33097503,0.614679}","{33082177,""ENSG00000142920"",0.466515,null,0.044839,4,1,506.041996}","{""chr1"",33081927,33082427,0,""+""}","{""chr1"",33081878,33082317,0,""."",33081878,33082179,""255,202,74"",""directionality:1|class:ELA"",1.0}","{390,0.78}"
0.194134,"""GSE267931_HUVEC_39C_120min""","""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",358,0.854415,"""TRIM62""","""FCHS_10021""","""cp3@TRIM62""","""PLA""","""genic""",false,84675.0,"{""chr1"",33097145,33097645}","{33097084,33097503,0.614679}","{33182070,""ENSG00000116525"",0.524194,null,0.013442,13,1,506.041996}","{""chr1"",33181820,33182320,0,""-""}","{""chr1"",33181859,33182582,0,""."",33182047,33182188,""0,0,204"",""directionality:-0.999247|class:PLA"",-0.999247}","{461,0.922}"
0.291194,"""GSE173536_lymphblastoid_d21_42C_60min""","""FCHS_100026""","""FCHS_100026""","""chr12""","""ELA""",201,1.0,"""RASSF9""","""FCHS_100044""","""cp1@RASSF9""","""PLA""","""intergenic""",false,411922.0,"{""chr12"",85423911,85425325}","{85423957,85424158,0.5}","{85836540,""ENSG00000198774"",0.357834,null,0.00446,9,1,0.0}","{""chr12"",85836290,85836790,0,""-""}","{""chr12"",85836188,85836732,0,""."",85836213,85836732,""0,0,255"",""directionality:-1|class:PLA"",-1.0}","{442,0.884}"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.644669,"""GSE101731_HFF_44C_60min""","""FCHS_99987""","""cp1@TSPAN19""","""chr12""","""PLA""",423,1.0,"""LRRIQ1""","""FCHS_99993""","""cp4@LRRIQ1,cp3@TSPAN19""","""PLA""","""genic""",false,19724.0,"{""chr12"",85016346,85016846}","{85016374,85016797,-0.977465}","{85036320,""ENSG00000133640"",0.470943,null,0.036875,2,2,0.0}","{""chr12"",85036070,85036570,0,""+""}","{""chr12"",85036236,85036498,0,""."",85036260,85036344,""204,0,0"",""directionality:0.970459|class:PLA"",0.970459}","{262,0.524}"
0.311435,"""GSE101731_HFF_44C_120min""","""FCHS_99987""","""cp1@TSPAN19""","""chr12""","""PLA""",423,1.0,"""SLC6A15""","""FCHS_99974""","""cp9@SLC6A15""","""PLA""","""genic""",false,103772.0,"{""chr12"",85016351,85016851}","{85016374,85016797,-0.977465}","{84912829,""ENSG00000072041"",0.486403,null,0.014404,3,1,0.0}","{""chr12"",84912579,84913079,0,""-""}","{""chr12"",84912565,84913337,0,""."",84912757,84912903,""0,0,204"",""directionality:-0.99533|class:PLA"",-0.99533}","{500,1.0}"
0.291244,"""GSE101731_HFF_44C_60min""","""FCHS_99987""","""cp1@TSPA

In [3]:
e2g_join_filter_unique_count = e2g_join_filter.group_by(
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "target_cre_id",
    "target_cre_name",
    "target_fantabio_cre_class"
).agg(
    pl.col("re2g_dataset").unique().alias("re2g_dataset"),
    pl.col("re2g_dataset").n_unique().alias("re2g_dataset_count"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").alias("enhancer_cre_overlap_frac_of_fantabio_cre"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").min().alias("enhancer_cre_overlap_frac_of_fantabio_cre_min"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").max().alias("enhancer_cre_overlap_frac_of_fantabio_cre_max"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").mean().alias("enhancer_cre_overlap_frac_of_fantabio_cre_mean"), # 該当するデータセットで同定された enhancer と fantabio の CRE の重複率の平均値

    pl.col("ENCODE-rE2G.Score").alias("ENCODE-rE2G.Score"),
    pl.col("ENCODE-rE2G.Score").min().alias("ENCODE-rE2G.Score_min"),
    pl.col("ENCODE-rE2G.Score").max().alias("ENCODE-rE2G.Score_max"),
    pl.col("ENCODE-rE2G.Score").mean().alias("ENCODE-rE2G.Score_mean"), # 該当するデータセットで同定された enhancer と target promoter の相互作用のスコアの平均値

    # pl.col("ENCODE-rE2G.Score").quantile(0.25).alias("ENCODE-rE2G.Score_q1"),
    # pl.col("ENCODE-rE2G.Score").median().alias("ENCODE-rE2G.Score_median"),
    # pl.col("ENCODE-rE2G.Score").quantile(0.75).alias("ENCODE-rE2G.Score_q3"),
    # pl.col("ENCODE-rE2G.Score").std().alias("ENCODE-rE2G.Score_std"),
).sort(
    by=["re2g_dataset_count"],
    descending=True
)

print(f"ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level): {e2g_join_filter_unique_count.select(['cre_id', 'target_cre_id']).n_unique()}")
display(e2g_join_filter_unique_count)

ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level): 368398


cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64
"""FCHS_202036""","""FCHS_202036""","""ELA""","""FCHS_202035""","""cp14@WIZ""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[1.0, 0.845324, … 1.0]",0.845324,1.0,0.987924,"[0.932402, 0.896114, … 0.685487]",0.441195,0.988182,0.862936
"""FCHS_500039""","""FCHS_500039""","""ELA""","""FCHS_162686""","""cp11@SLC7A5""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[1.0, 1.0, … 1.0]",1.0,1.0,1.0,"[0.610364, 0.807375, … 0.867379]",0.267399,0.905857,0.748022
"""FCHS_118481""","""FCHS_118481""","""PLA""","""FCHS_118549""","""cp10@RASA3""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[1.0, 0.685637, … 0.677507]",0.654472,1.0,0.881533,"[0.914299, 0.920479, … 0.850441]",0.352148,0.920479,0.674624
"""FCHS_355957""","""FCHS_355957""","""PLA""","""FCHS_355932""","""cp11@FKBP5""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.552674, 0.585089, … 1.0]",0.552674,1.0,0.922783,"[0.181076, 0.476446, … 0.225056]",0.181076,0.835387,0.568878
"""FCHS_393692""","""cp1@RNY3""","""ELA""","""FCHS_393695""","""cp1@GHET1,cp2@RNY1""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[1.0, 1.0, … 1.0]",1.0,1.0,1.0,"[0.931632, 0.891956, … 0.906371]",0.771432,0.969518,0.897168
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_197608""","""FCHS_197608""","""ELA""","""FCHS_197611""","""cp2@TJP3""","""PLA""","[""GSE267931_HUVEC_39C_120min""]",1,[0.623218],0.623218,0.623218,0.623218,[0.376168],0.376168,0.376168,0.376168
"""FCHS_197052""","""FCHS_197052""","""ELA""","""FCHS_196317""","""cp1@PCSK4,cp2@REEP6""","""PLA""","[""GSE105028_hESC_42C_60min""]",1,[1.0],1.0,1.0,1.0,[0.328062],0.328062,0.328062,0.328062
"""FCHS_288850""","""cp19@FOXP1""","""PLA""","""FCHS_289270""","""cp1@ZNF717""","""PLA""","[""GSE105028_hESC_42C_60min""]",1,[0.907635],0.907635,0.907635,0.907635,[0.538279],0.538279,0.538279,0.538279


&nbsp;

&nbsp;

&nbsp;

## 2. up5p transcript list

In [4]:
up5p_fantabio = pl.read_parquet(
    "../../out/fantabio/parquet_file/up5p_transcript_fantabio_v261001.parquet"
)

pl.Config.set_tbl_rows(2)
display(up5p_fantabio)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_28105""","""chr1""",161523745,"""161524487""","""cp1@HSPA6""","[""GQ891451""]","""0""",53,"[{""EH38E2843853"",""pELS""}, {""EH38E2843855"",""PLS""}, {""EH38E3981533"",""pELS""}]","[{""STAG2"",4103,[{""SRX9975614"",4103}, {""SRX9975613"",3500}, {""SRX26159208"",562}]}, {""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX13845684"",554}]}, … {""GABPA"",500,[{""SRX2635792"",500}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,0.982111,"""PLA""","""204,0,0""","""red"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000774581.1""",101,57,248,44,34,44,173,-10,null,null,"""lncRNA""","""ENSG00000300853.1""","[""basic"", ""TAGENE""]","""+""",2,"""URS0002A25289""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_400104""","""chr8""",27592922,"""27593122""","""FCHS_400104""","[""ENST00000774579.1""]","""0""",294,"[{""EH38E3824983"",""dELS""}]",null,"[{""HS_00105821"",""NHEK-Epidermal Keratinocyte Primary Cells"",""Skin"",""Skin"",""SVM""}, {""HS_00105822"",""H1 BMP4 Derived Trophoblast Cultured Cells"",""Trophoblast cell"",""Others"",""SVM""}, … {""HS_00105828"",""Breast variant Human Mammary Epithelial Cells (vHMEC)"",""Breast"",""Breast"",""SVM""}]",null,null,0.5,"""ELA""","""255,202,74""","""yellow"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 3. Selection of transcript (UP-UP)

- 上記のセルで読み込んだ `up5p_fantabio` データフレームから、enhancer 側の transcript と、target promoter 側の transcript をそれぞれ candidate として選択する
- enhancer 側の transcript と、target promoter 側の transcript はこの時点では同じ数になり、join することによってフィルタリングしていく

In [5]:
# enhancer side transcript entry (enhancer potential transcript - CRE pair)
fantabio_HN2_up5p_enhancer_transcript_candidate = up5p_fantabio.select(
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "transcript_id",
    "HN2",
    "transcript_type",
    "gene_id",
    "gene_symbol",
    "strand",
    "RNACentral_ID",
    "annotation_remarks",
    "tags"
).unique().rename(
    # assign frag name "e" to enhancer side
    {
        "transcript_id": "e_transcript_id",
        "HN2": "e_HN2",
        "transcript_type": "e_transcript_type",
        "gene_id": "e_gene_id",
        "gene_symbol": "e_gene_symbol",
        "strand": "e_strand",
        "RNACentral_ID": "e_RNACentral_ID",
        "annotation_remarks": "e_annotation_remarks",
        "tags": "e_tags"
    }
)

display(fantabio_HN2_up5p_enhancer_transcript_candidate)

cre_id,cre_name,fantabio_cre_class,e_transcript_id,e_HN2,e_transcript_type,e_gene_id,e_gene_symbol,e_strand,e_RNACentral_ID,e_annotation_remarks,e_tags
str,str,str,str,i64,str,str,str,str,str,list[str],list[str]
"""FCHS_202191""","""cp4@RAB8A""","""PLA""","""ENST00000860860.1""",63,"""protein_coding""","""ENSG00000167461.13""","""RAB8A""","""+""",null,"[""DanaFarber""]","[""basic"", ""TAGENE""]"
…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_356003""","""cp3@SRPK1""","""PLA""","""ENST00000510290.5""",70,"""protein_coding""","""ENSG00000096063.17""","""SRPK1""","""-""",null,null,"[""mRNA_end_NF"", ""cds_end_NF""]"


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## JOIN (1)

- 標的となる promoter の近傍にある transcript は一旦統合せず、enhancer RNA の potential がある transcript だけを一旦出力する

In [6]:
e2g_join_filter_unique_count_join = e2g_join_filter_unique_count.join(
    fantabio_HN2_up5p_enhancer_transcript_candidate,
    on=["cre_id", "cre_name", "fantabio_cre_class"],
    how="inner"
).filter(
    pl.col("e_transcript_type") == "lncRNA" # lncRNA のみを対象とする
)


display(e2g_join_filter_unique_count_join)

cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean,e_transcript_id,e_HN2,e_transcript_type,e_gene_id,e_gene_symbol,e_strand,e_RNACentral_ID,e_annotation_remarks,e_tags
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64,str,i64,str,str,str,str,str,list[str],list[str]
"""FCHS_94451""","""FCHS_94451""","""PLA""","""FCHS_94499""","""cp1@ATG101""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.778816, 0.778816, … 0.904984]",0.767913,0.904984,0.806297,"[0.355003, 0.189876, … 0.526484]",0.189876,0.774307,0.408416,"""ENST00000812391.1""",128,"""lncRNA""","""ENSG00000305684.1""",null,"""-""","""URS0002A236EE""","[""CLS3 project""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_265630""","""cp2@LOC105369306""","""ELA""","""FCHS_265830""","""cp4@RIPPLY3""","""PLA""","[""GSE105028_hESC_42C_60min""]",1,[1.0],1.0,1.0,1.0,[0.27554],0.27554,0.27554,0.27554,"""ENST00000422473.2""",51,"""lncRNA""","""ENSG00000233393.3""",null,"""+""","""URS0002A57053""","[""CLS3 project""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]"


In [7]:
e2g_join_filter_unique_count_join.write_parquet(
    "../../out/transcript_classification/enhancer_derived_lncRNA/up5p_enhancer_derived_lncRNA_v261010.parquet"
)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Join (2) (UP (enhancer proximal lncRNA) - UP (target promoter proximal transcript))

- このプロセスで `join`することで得られるのは、enhancer - promoter 両方の近傍にある transcript が 発現上昇したとみなしている transcirpt
- 予測された enhancer 側 で、`PLA` or `ELA` でも フィルタリングしないことに決定

In [8]:
# target promoter side transcript candidate
fantabio_HN2_up5p_target_transcript_candidate = up5p_fantabio.select(
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "transcript_id",
    "HN2",
    "transcript_type",
    "gene_id",
    "gene_symbol",
    "strand",
    "RNACentral_ID",
    "annotation_remarks",
    "tags"
).unique().rename(
    # assign frag name "t" to target promoter side
    {
        "cre_id": "target_cre_id",
        "cre_name": "target_cre_name",
        "fantabio_cre_class": "target_fantabio_cre_class",
        "transcript_id": "t_transcript_id",
        "HN2": "t_HN2",
        "transcript_type": "t_transcript_type",
        "gene_id": "t_gene_id",
        "gene_symbol": "t_gene_symbol",
        "strand": "t_strand",
        "RNACentral_ID": "t_RNACentral_ID",
        "annotation_remarks": "t_annotation_remarks",
        "tags": "t_tags"
    }
)

display(fantabio_HN2_up5p_target_transcript_candidate)

target_cre_id,target_cre_name,target_fantabio_cre_class,t_transcript_id,t_HN2,t_transcript_type,t_gene_id,t_gene_symbol,t_strand,t_RNACentral_ID,t_annotation_remarks,t_tags
str,str,str,str,i64,str,str,str,str,str,list[str],list[str]
"""FCHS_112460""","""cp3@TPT1,cp3@TPT1-AS1""","""PLA""","""ENST00000662212.1""",44,"""retained_intron""","""ENSG00000170919.17""","""TPT1-AS1""","""+""",null,"[""Assembled from SLRseq reads (SRP049776)""]","[""TAGENE""]"
…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_123976""","""cp1@JKAMP""","""ELA""","""ENST00000554754.5""",70,"""protein_coding_CDS_not_defined""","""ENSG00000050130.19""","""JKAMP""","""+""",null,null,[]


In [9]:
e2g_join_filter_unique_count_join = e2g_join_filter_unique_count.join(
    fantabio_HN2_up5p_enhancer_transcript_candidate,
    on=["cre_id", "cre_name", "fantabio_cre_class"],
    how="inner"
).join(
    fantabio_HN2_up5p_target_transcript_candidate,
    on=["target_cre_id", "target_cre_name", "target_fantabio_cre_class"],
    how="inner"
).sort(
    by=["e_HN2", "t_HN2", "re2g_dataset_count"],
    descending=[True, True, True]
).filter(
    pl.col("e_transcript_type") == "lncRNA" # lncRNA のみを対象とする
)

e2g_join_filter_unique_count_join.write_parquet(
    "../../out/transcript_classification/enhancer_derived_lncRNA/up5p_enhancer_derived_lncRNA_v261010_UP_UP.parquet"
)

display(e2g_join_filter_unique_count_join)

cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean,e_transcript_id,e_HN2,e_transcript_type,e_gene_id,e_gene_symbol,e_strand,e_RNACentral_ID,e_annotation_remarks,e_tags,t_transcript_id,t_HN2,t_transcript_type,t_gene_id,t_gene_symbol,t_strand,t_RNACentral_ID,t_annotation_remarks,t_tags
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64,str,i64,str,str,str,str,str,list[str],list[str],str,i64,str,str,str,str,str,list[str],list[str]
"""FCHS_28129""","""cp1@HSPA7""","""ELA""","""FCHS_28299""","""cp4@UAP1,cp1@UAP1-DT""","""PLA""","[""GSE105028_hESC_42C_60min""]",1,[1.0],1.0,1.0,1.0,[0.810275],0.810275,0.810275,0.810275,"""ENST00000850140.1""",311,"""lncRNA""","""ENSG00000310471.1""",null,"""+""","""URS0002A15385""","[""CLS3 project"", ""overlaps pseudogene""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""ENST00000898008.1""",105,"""protein_coding""","""ENSG00000117143.15""","""UAP1""","""+""",null,"[""DanaFarber""]","[""basic"", ""TAGENE"", ""CCDS""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_231344""","""FCHS_231344""","""ELA""","""FCHS_231321""","""cp1@ZNG1B""","""PLA""","[""GSE280307_THP1_42C_120min""]",1,[0.671271],0.671271,0.671271,0.671271,[0.234079],0.234079,0.234079,0.234079,"""ENST00000801518.1""",44,"""lncRNA""","""ENSG00000310523.1""",null,"""+""","""URS0002A559D4""","[""CLS3 project""]","[""TAGENE""]","""ENST00000947888.1""",44,"""protein_coding""","""ENSG00000136682.17""","""ZNG1B""","""+""",null,"[""ENCODE_tissue""]","[""basic"", ""TAGENE""]"


In [10]:
# check (ENEMAL)
ENST00000701343_2 = e2g_join_filter_unique_count_join.filter(
    pl.col("e_gene_symbol") == "ENEMAL"
)

pl.Config.set_tbl_rows(2)
display(ENST00000701343_2)

cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean,e_transcript_id,e_HN2,e_transcript_type,e_gene_id,e_gene_symbol,e_strand,e_RNACentral_ID,e_annotation_remarks,e_tags,t_transcript_id,t_HN2,t_transcript_type,t_gene_id,t_gene_symbol,t_strand,t_RNACentral_ID,t_annotation_remarks,t_tags
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64,str,i64,str,str,str,str,str,list[str],list[str],str,i64,str,str,str,str,str,list[str],list[str]
"""FCHS_73648""","""cp3@ENEMAL""","""ELA""","""FCHS_73856""","""cp6@SIPA1""","""PLA""","[""GSE105028_hESC_42C_60min""]",1,[1.0],1.0,1.0,1.0,[0.297571],0.297571,0.297571,0.297571,"""ENST00000701343.2""",81,"""lncRNA""","""ENSG00000289883.2""","""ENEMAL""","""-""","""URS0002A39AA9""","[""CLS3 project"", ""Assembled from PacBio CLS3 reads""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""ENST00000969244.1""",163,"""protein_coding""","""ENSG00000213445.12""","""SIPA1""","""+""",null,"[""ENCODE_tissue""]","[""basic"", ""GENCODE_Primary"", ""TAGENE""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_73651""","""cp6@ENEMAL""","""ELA""","""FCHS_470991""","""cp8@TALAM1""","""PLA""","[""GSE173536_lymphblastoid_d21_42C_60min""]",1,[0.878906],0.878906,0.878906,0.878906,[0.195157],0.195157,0.195157,0.195157,"""ENST00000701343.2""",81,"""lncRNA""","""ENSG00000289883.2""","""ENEMAL""","""-""","""URS0002A39AA9""","[""CLS3 project"", ""Assembled from PacBio CLS3 reads""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""ENST00000698129.2""",45,"""lncRNA""","""ENSG00000289740.2""","""TALAM1""","""-""","""URS0002A4D8F6""","[""NR_145459.1 is the basis for this transcript. Built in response to a GENCODE ticket as RefSeq have this gene annotated. Discovered as RNA using coprecipitation and then characterised using strand specific PCR. There are two papers which deal with this gene but could not find any sequence with which to define the 5' and 3' ends.(PMID: 31382922) (PMID: 26826711). We got the raw RACE reads which I mapped and they looked OK"", ""CLS3 project""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]"


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 5. Join (UP (enhancer proximal lncRNA) - DOWN (target promoter proximal transcript))

- `silencer` 的な動きをしている enhancer-derived lncRNA も探索する！

In [11]:
down5p_fantabio = pl.read_parquet(
    "../../out/fantabio/parquet_file/bottom5p_transcript_fantabio_v261001.parquet"
)

display(down5p_fantabio)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000834029.1""",69,117,220,-48,47,56,148,-9,"""EXO5-DT""","""HGNC:55647""","""lncRNA""","""ENSG00000238287.2""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""-""",4,"""URS0002A1647B""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_11684""","""chr1""",40508315,"""40508650""","""cp5@EXO5""","[""ENST00000925740.1""]","""0""",-63,"[{""EH38E2805480"",""PLS""}]","[{""ZNF687"",3399,[{""SRX10478566"",3399}, {""SRX10478565"",2503}]}, {""TCF7"",2966,[{""SRX10774913"",2966}, {""SRX10774914"",2622}]}, … {""NFATC1"",515,[{""SRX335071"",515}]}]","[{""HS_00004908"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}, {""HS_00004909"",""Foreskin Melanocyte Primary Cells skin01"",""Prepuce"",""Skin"",""SVM""}, … {""HS_00004912"",""Small Intestine"",""Small intestine"",""Intestine"",""SVM""}]","[{""hg19::chr1:40974317..40974346,-;hg_11928.1"",""hg_11928.1""}]",null,0.952663,"""ELA""","""255,202,74""","""yellow"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000549545.1""",11,265,130,-254,8,174,69,-166,"""COQ10A""","""HGNC:26515""","""retained_intron""","""ENSG00000135469.15""",[],"""+""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_96252""","""chr12""",56268140,"""56268446""","""cp6@COQ10A""","[""ENST00000549545.1""]","""-23""",-23,"[{""EH38E4064794"",""pELS""}]",null,"[{""HS_00173371"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_02053667"",""Thyroid Gland"",""Thyroid gland"",""Others"",""gkmSVM""}]",null,null,-0.368421,"""ELA""","""255,202,74""","""yellow"""


In [12]:
# target promoter side transcript candidate
fantabio_HN2_down5p_target_transcript_candidate = down5p_fantabio.select(
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "transcript_id",
    "HN2",
    "transcript_type",
    "gene_id",
    "gene_symbol",
    "strand",
    "RNACentral_ID",
    "annotation_remarks",
    "tags"
).unique().rename(
    # assign frag name "t" to target promoter side
    {
        "cre_id": "target_cre_id",
        "cre_name": "target_cre_name",
        "fantabio_cre_class": "target_fantabio_cre_class",
        "transcript_id": "t_transcript_id",
        "HN2": "t_HN2",
        "transcript_type": "t_transcript_type",
        "gene_id": "t_gene_id",
        "gene_symbol": "t_gene_symbol",
        "strand": "t_strand",
        "RNACentral_ID": "t_RNACentral_ID",
        "annotation_remarks": "t_annotation_remarks",
        "tags": "t_tags"
    }
)

display(fantabio_HN2_down5p_target_transcript_candidate)

target_cre_id,target_cre_name,target_fantabio_cre_class,t_transcript_id,t_HN2,t_transcript_type,t_gene_id,t_gene_symbol,t_strand,t_RNACentral_ID,t_annotation_remarks,t_tags
str,str,str,str,i64,str,str,str,str,str,list[str],list[str]
"""FCHS_20029""","""cp3@SASS6""","""ELA""","""ENST00000482437.5""",-127,"""nonsense_mediated_decay""","""ENSG00000122435.12""","""TRMT13""","""+""",null,null,[]
…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_380303""","""cp1@UBE2D4,cp12@URGCP""","""PLA""","""ENST00000426198.5""",-59,"""protein_coding""","""ENSG00000106608.18""","""URGCP""","""-""",null,"[""alternative 5' UTR""]","[""mRNA_end_NF"", ""cds_end_NF"", ""alternative_5_UTR""]"


In [13]:
e2g_join_filter_unique_count_join_up_down = e2g_join_filter_unique_count.join(
    fantabio_HN2_up5p_enhancer_transcript_candidate,
    on=["cre_id", "cre_name", "fantabio_cre_class"],
    how="inner"
).join(
    fantabio_HN2_down5p_target_transcript_candidate,
    on=["target_cre_id", "target_cre_name", "target_fantabio_cre_class"],
    how="inner"
).sort(
    by=["e_HN2", "t_HN2", "re2g_dataset_count"],
    descending=[True, True, True]
).filter(
    pl.col("e_transcript_type") == "lncRNA" # lncRNA のみを対象とする
)

e2g_join_filter_unique_count_join_up_down.write_parquet(
    "../../out/transcript_classification/enhancer_derived_lncRNA/up5p_enhancer_derived_lncRNA_v261010_UP_DOWN.parquet"
)

display(e2g_join_filter_unique_count_join_up_down)

cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean,e_transcript_id,e_HN2,e_transcript_type,e_gene_id,e_gene_symbol,e_strand,e_RNACentral_ID,e_annotation_remarks,e_tags,t_transcript_id,t_HN2,t_transcript_type,t_gene_id,t_gene_symbol,t_strand,t_RNACentral_ID,t_annotation_remarks,t_tags
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64,str,i64,str,str,str,str,str,list[str],list[str],str,i64,str,str,str,str,str,list[str],list[str]
"""FCHS_28129""","""cp1@HSPA7""","""ELA""","""FCHS_28299""","""cp4@UAP1,cp1@UAP1-DT""","""PLA""","[""GSE105028_hESC_42C_60min""]",1,[1.0],1.0,1.0,1.0,[0.810275],0.810275,0.810275,0.810275,"""ENST00000850140.1""",311,"""lncRNA""","""ENSG00000310471.1""",null,"""+""","""URS0002A15385""","[""CLS3 project"", ""overlaps pseudogene""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""ENST00000898018.1""",-55,"""protein_coding""","""ENSG00000117143.15""","""UAP1""","""+""",null,"[""DanaFarber""]","[""basic"", ""TAGENE"", ""appris_alternative_1""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_158494""","""FCHS_158494""","""ELA""","""FCHS_158518""","""FCHS_158518""","""ELA""","[""GSE267931_HUVEC_39C_120min""]",1,[1.0],1.0,1.0,1.0,[0.3979],0.3979,0.3979,0.3979,"""ENST00000702375.2""",44,"""lncRNA""","""ENSG00000289991.2""",null,"""+""","""URS0002A3F8CA""","[""Assembled from PacBio CLS3 reads"", ""CLS3 project""]","[""TAGENE""]","""ENST00000332395.7""",-219,"""protein_coding""","""ENSG00000182810.7""","""DDX28""","""-""",null,"[""MANE_select""]","[""basic"", ""Ensembl_canonical"", … ""CCDS""]"
